In [4]:
import numpy as np
import pandas as pd
from scipy.signal import detrend, savgol_filter, butter, filtfilt
from scipy.fft import rfft, rfftfreq

# ============================================================
# LOAD DATA
# ============================================================

def load_data():

    lift = pd.read_csv(
        "drag_lift_body_001.dat",
        sep=r"\s+",
        skiprows=1,
        names=["time","cxp","cxs","cx","cyp","cys","cy"]
    )

    struct = pd.read_csv(
        "viv_body1_probe.dat",
        sep=r"\s+",
        skiprows=2,
        names=["ntime","time","DX","DY","VY","AY"]
    )

    t = struct.time.values
    y = struct.DY.values

    # remove transient
    mask = t > 50.0
    t = t[mask]
    y = y[mask]

    # interpolate lift
    F = np.interp(t, lift.time.values, lift.cy.values)

    t = t - t[0]

    return t, y, F


# ============================================================
# FREQUENCY EXTRACTION
# ============================================================

def compute_frequency(t, y):

    y = detrend(y)

    dt = np.mean(np.diff(t))
    N = len(t)

    Y = rfft(y)
    freqs = rfftfreq(N, dt)

    Y[0] = 0

    idx = np.argmax(np.abs(Y))
    f = freqs[idx]

    omega = 2*np.pi*f
    T = 1/f

    return f, omega, T


# ============================================================
# LOWPASS FILTER (PRESERVES NONLINEARITIES)
# ============================================================

def lowpass(x, fs, f0):

    # Lowpass filter at 5 times the fundamental frequency to keep nonlinear harmonics
    # while removing high frequency numerical noise
    cutoff = 5.0 * f0
    b, a = butter(4, cutoff / (fs/2), btype='low')
    return filtfilt(b, a, x)


# ============================================================
#  WAKE VARIABLE
# ============================================================

def compute_q(F):

    F = detrend(F)

    CL0 = 0.328

    q = 2 * F / CL0

    return q


# ============================================================
# IDENTIFICATION (ROBUST)
# ============================================================

def identify_wake_params(t, y, q, omega):

    dt = np.mean(np.diff(t))

    # Directly compute derivatives using Savitzky-Golay
    y_tt = savgol_filter(y, window_length=101, polyorder=3, deriv=2, delta=dt)
    q_t  = savgol_filter(q, window_length=101, polyorder=3, deriv=1, delta=dt)
    q_tt = savgol_filter(q, window_length=101, polyorder=3, deriv=2, delta=dt)

    # Standard Facchinetti et al. form:
    # q'' + eps * omega * (q^2 - 1) * q' + omega^2 * q = A * y''
    # Rearranged for regression:
    # -q'' - omega^2 * q = eps * [omega * (q^2 - 1) * q'] + A * [-y'']

    X1 = omega * (q**2 - 1) * q_t
    X2 = -y_tt
    Y  = -q_tt - (omega**2) * q

    # global scaling (preserves physics, helps matrix conditioning)
    scale = np.std(Y)
    X = np.vstack([X1, X2]).T / scale
    Y = Y / scale

    # ridge regression
    lam = 1e-4
    XtX = X.T @ X + lam * np.eye(2)
    XtY = X.T @ Y

    coeffs = np.linalg.solve(XtX, XtY)

    eps = coeffs[0]
    A   = coeffs[1]

    return eps, A


# ============================================================
# MAIN
# ============================================================

def main():

    print("\n===== VIV SYSTEM IDENTIFICATION (FINAL) =====")

    t, y, F = load_data()

    # --------------------------------------------------------
    # MASS
    # --------------------------------------------------------
    m_total = 5.0

    # --------------------------------------------------------
    # FREQUENCY
    # --------------------------------------------------------
    f, omega, T = compute_frequency(t, y)

    # --------------------------------------------------------
    # FILTER SIGNALS
    # --------------------------------------------------------
    fs = 1 / np.mean(np.diff(t))

    y = lowpass(y, fs, f)
    F = lowpass(F, fs, f)

    # --------------------------------------------------------
    # STRUCTURAL PARAMETERS
    # --------------------------------------------------------
    k = m_total * omega**2

    zeta = 0.01
    c = 2 * zeta * m_total * omega

    # --------------------------------------------------------
    # WAKE VARIABLE (FIXED)
    # --------------------------------------------------------
    q = compute_q(F)

    # --------------------------------------------------------
    # IDENTIFICATION
    # --------------------------------------------------------
    eps, A = identify_wake_params(t, y, q, omega)

    # ========================================================
    # OUTPUT
    # ========================================================

    print("\n================ RESULTS ================")
    print(f"Frequency f        : {f:.5f} Hz")
    print(f"Angular freq ω     : {omega:.5f} rad/s")
    print(f"Period T           : {T:.5f} s")
    print("----------------------------------------")
    print(f"Mass (m)           : {m_total:.5f}")
    print(f"Stiffness (k)      : {k:.5f}")
    print(f"Damping (c)        : {c:.5f}")
    print(f"Epsilon (ε)        : {eps:.5f}")
    print(f"Coupling (A)       : {A:.5f}")
    print("========================================")

    print("\n================ GOVERNING EQUATIONS =================")

    print("\nStructure equation:")
    print(f"{m_total:.3f} y'' + {c:.3f} y' + {k:.3f} y = F(t)")

    print("\nWake oscillator (Facchinetti Standard Form):")
    print(f"q'' + {eps:.4f} * {omega:.3f} * (q^2 - 1)q' + {omega**2:.3f} q = {A:.3f} y''")

    print("\n=====================================================\n")


if __name__ == "__main__":
    main()


===== VIV SYSTEM IDENTIFICATION (FINAL) =====

================ RESULTS ================
Frequency f        : 0.16421 Hz
Angular freq ω     : 1.03177 rad/s
Period T           : 6.08974 s
----------------------------------------
Mass (m)           : 5.00000
Stiffness (k)      : 5.32270
Damping (c)        : 0.10318
Epsilon (ε)        : 0.00584
Coupling (A)       : -3.55500

================ GOVERNING EQUATIONS =================

Structure equation:
5.000 y'' + 0.103 y' + 5.323 y = F(t)

Wake oscillator (Facchinetti Standard Form):
q'' + 0.0058 * 1.032 * (q^2 - 1)q' + 1.065 q = -3.555 y''


